### Agent

<small>User prompt: "Schedule a meeting with Joe next week and send him a confirmation email."

To complete this, the LLM with help of Agent will do the following tasks:  
- Check your calendar
- Find free time slots
- Check Joe’s availability
- Create the calendar event
- Send a confirmation email
- Ask follow-up questions if information is missing</small>

### Chain vs Agent

<small>

**A Chain** is a fixed sequence of LLM calls. You define every step in code before the task runs.  
**An Agent** decides dynamically which steps to take based on what it observes at runtime.

| | Chain | Agent |
|---|---|---|
| **Steps** | Fixed, defined upfront | Dynamic, chosen by the LLM |
| **Decision maker** | You (the programmer) | The LLM |
| **Tools** | Called in a fixed order | Called based on need |
| **Best for** | Predictable pipelines | Open-ended, conditional tasks |

---

**Finance example**
- *Chain*: "Summarise this earnings report" → fixed prompt → fixed output. You already know the path.
- *Agent*: "Is my portfolio overexposed to tech?" → checks current holdings → checks sector weights → checks risk threshold → re-plans if data is missing → proposes rebalancing. The path depends on what each step returns.

**Education example**
- *Chain*: "Grade this essay against this rubric" → rubric + essay → score. One step, predictable.
- *Agent*: "What courses should I take next semester?" → checks transcript → verifies prerequisites → detects schedule conflicts → ranks options. Cannot plan the sequence until step 1 completes.

**Healthcare example**
- *Chain*: "Summarise this lab report for the patient" → template + report → output. Fixed path.
- *Agent*: "Book the earliest available cardiology appointment" → checks doctor availability → checks patient calendar → handles conflicts → confirms booking. Each step depends on the previous result.

</small>

**Agent Loop**  
<small>It repeatedly performs:  
- Reason  
- Choose an action  
- Call a tool  
- Observe result  
- Update plan  
- Repeat until done</nsmall>

#### When to Use Agents vs Chains vs Direct Calls

<small>

| Scenario | Right tool | Why |
|---|---|---|
| Translate a document | Direct API call | Single step, no decisions needed |
| Classify customer support emails | Chain | Fixed input → label, predictable path |
| Answer a product FAQ | Chain (RAG) | Retrieve + generate, known structure |
| "Plan my investment strategy" | Agent | Unknown number of steps, conditional decisions |
| "Enrol me in the best ML course for my level" | Agent | Must check level, prerequisites, seat availability |
| "Monitor portfolio and alert on drops > 5%" | Agent + loop | Ongoing observation + conditional action |

---

**Agents are overkill when:**
- The path from input to output never changes
- There are fewer than 3 steps and no branching
- The task does not depend on what earlier steps returned
- Latency or cost is a hard constraint

**Finance overkill**: Converting USD to EUR at a fixed rate — a single function call, not an agent.  
**Education overkill**: Marking a multiple-choice quiz — deterministic, no tool selection needed.

**Rule of thumb:**  
If you can draw the complete flowchart *before* running the task, use a chain.  
If the flowchart depends on what the agent discovers, use an agent.

</small>

### ReAct pattern(Reason + Act)

```mermaid
---
title: ReAct Pattern
---
flowchart LR
    A[User Query] --> B(REason \n Analyze & Decide)
    B --> C>Task Complete?]
    C -->|Yes| D[Final Result]
    C -->|No| E[ACTion \nUse Tools & Execute]
    E --> F[Observation \n review result]
    F --> B
```

<small>ReAct pattern combines:  
- Reasoning steps (“Thought”)  
- Actions (“Tool Calls”)  
- Observations (“Tool Results”)  

**Typical Flow**:  

Thought: I need weather data  
Action: call_weather_api("Bangalore")  
Observation: Rainy, 24°C  

Thought: Need calendar availability  
Action: call_calendar_api()  
Observation: Free at 4 PM  

Thought: Enough information gathered  
Final Answer: Schedule outdoor meeting tomorrow instead  

The model alternates between:  
Internal reasoning  
External actions</nsmall>

### Tools

<small>Tools are external functions/APIs the agent can use.  
Examples:  
Weather API  
Calendar API   
Search engine   
Email sender</nsmall>

<small>How LLM selects the appropriate tool?  
The LLM reads:   
- User request  
- Available tool descriptions  
- Previous observations  

Then predicts: Which tool best helps accomplish the task?</nsmall>

<small>Available Tools
| Tool           | Purpose               |
| -------------- | --------------------- |
| `search_web`   | Search internet       |
| `send_email`   | Send email            |
| `create_event` | Create calendar event |</small>


#### Domain-specific Tool Schemas

<small>The same JSON schema structure applies across all domains.  
Notice how the `description` field is the primary signal the LLM uses to decide *when* to call each tool.  
High-stakes tools (booking, trading, enrolment) should flag this in their description so the agent can apply an approval gate.</small>

In [ ]:
import json

# Finance: portfolio lookup
portfolio_schema = {
    "name": "get_portfolio",
    "description": "Return current holdings and sector weights for a user's investment portfolio.",
    "parameters": {
        "type": "object",
        "properties": {
            "user_id": {"type": "string", "description": "Unique investor account ID"},
            "include_performance": {
                "type": "boolean",
                "description": "Include YTD return figures",
                "default": False,
            },
        },
        "required": ["user_id"],
    },
}

# Education: course search
course_search_schema = {
    "name": "search_courses",
    "description": "Search available courses by topic, difficulty level, or prerequisite.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "Subject area, e.g. 'machine learning'"},
            "level": {
                "type": "string",
                "enum": ["beginner", "intermediate", "advanced"],
            },
            "prerequisites_met": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of course codes the student has already completed",
            },
        },
        "required": ["topic"],
    },
}

# Healthcare: appointment booking (high-stakes — requires approval)
appointment_schema = {
    "name": "book_appointment",
    "description": "Book a medical appointment. HIGH-STAKES: requires human approval before execution.",
    "parameters": {
        "type": "object",
        "properties": {
            "patient_id": {"type": "string"},
            "doctor_specialty": {"type": "string", "description": "e.g. 'cardiology', 'general'"},
            "preferred_date": {"type": "string", "format": "date", "description": "ISO 8601 date"},
            "reason": {"type": "string", "description": "Brief clinical reason for visit"},
        },
        "required": ["patient_id", "doctor_specialty"],
    },
}

# E-commerce: order status lookup
order_status_schema = {
    "name": "get_order_status",
    "description": "Retrieve current status and estimated delivery for a customer order.",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},
            "include_history": {"type": "boolean", "default": False},
        },
        "required": ["order_id"],
    },
}

for schema in [portfolio_schema, course_search_schema, appointment_schema, order_status_schema]:
    print(f"\n{'='*50}")
    print(f"Domain: {schema['name']}")
    print(json.dumps(schema, indent=2))

JSON Tool Schema

In [1]:
import json

send_email_schema = {
    "name": "send_email",
    "description": "Send an email to a recipient",
    "parameters": {
        "type": "object",
        "properties": {
            "to": {"type": "string"},
            "subject": {"type": "string"},
            "body": {"type": "string"},
        },
        "required": ["to", "subject", "body"],
    },
}

print(json.dumps(send_email_schema, indent=2))

{
  "name": "send_email",
  "description": "Send an email to a recipient",
  "parameters": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string"
      },
      "subject": {
        "type": "string"
      },
      "body": {
        "type": "string"
      }
    },
    "required": [
      "to",
      "subject",
      "body"
    ]
  }
}


<small>This JSON schema tells the model exactly which fields the tool expects.  
The required keys are `to`, `subject`, and `body`, so the agent can validate arguments before calling the tool.</small>

<small> Using the available tools and their tool schemas the model decides the tools that are required and passes the appropriate arguments.  
send_email(  
&emsp;to="joe@example.com",  
&emsp;subject="Meeting Confirmed",  
&emsp;body="Your meeting is confirmed."  
)</small>

Tool creation and management

In [2]:
from abc import ABC, abstractmethod
import requests
import os

class Tool(ABC):
    name = ""
    description = ""

    @abstractmethod
    def run(self, **kwargs):
        raise NotImplementedError


In [3]:
class WeatherTool(Tool):
    name = "weather"
    description = (
        "Get current weather information for a city."
    )

    def __init__(self, api_key):
        self.api_key = api_key

    def run(self, city):
        if not self.api_key:
            return {
                "status": "success",
                "source": "local_mock",
                "city": city,
                "temperature_c": 24,
                "condition": "clear sky",
                "humidity": 48,
            }

        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": city,
            "appid": self.api_key,
            "units": "metric"
        }
        try:
            response = requests.get(url, params=params, timeout=5)
            if response.status_code != 200:
                return {
                    "status": "error",
                    "error_type": "unexpected_output",
                    "message": response.json().get("message", "weather lookup failed"),
                }
            data = response.json()
            return {
                "status": "success",
                "source": "api",
                "city": city,
                "temperature_c": data["main"]["temp"],
                "condition": data["weather"][0]["description"],
                "humidity": data["main"]["humidity"],
            }
        except Exception as exc:
            return {
                "status": "error",
                "error_type": "timeout",
                "message": str(exc),
            }

In [ ]:
class ExchangeRateTool(Tool):
    name = "exchange_rate"
    description = (
        "Get exchange rate between two currencies."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, base_currency, target_currency):
        url = (
            f"https://v6.exchangerate-api.com/v6/"
            f"{self.api_key}/latest/{base_currency.upper()}"
        )
        response = requests.get(url)
        if response.status_code != 200:
            return "Failed to fetch exchange rates."
        data = response.json()
        if data["result"] != "success":
            return data.get("error-type", "Unknown error")
        rates = data["conversion_rates"]
        if target_currency.upper() not in rates:
            return f"Currency '{target_currency}' not found."
        rate = rates[target_currency.upper()]
        return (
            f"Exchange Rate:\n"
            f"1 {base_currency.upper()} = "
            f"{rate} {target_currency.upper()}"
        )

In [ ]:
class NewsTool(Tool):
    name = "news"
    description = (
        "Get latest news headlines for a topic."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, topic="technology", limit=5):
        url = "https://newsapi.org/v2/everything"
        params = {
            "q": topic,
            "sortBy": "publishedAt",
            "language": "en",
            "pageSize": limit,
            "apiKey": self.api_key
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return "Failed to fetch news."
        data = response.json()
        if data["status"] != "ok":
            return data.get("message", "Unknown error")
        articles = data["articles"]
        if not articles:
            return f"No news found for '{topic}'."
        result = [f"Latest news about {topic}:\n"]
        for i, article in enumerate(articles, start=1):
            result.append(
                f"{i}. {article['title']}\n"
                f"   Source: {article['source']['name']}\n"
                f"   URL: {article['url']}\n"
            )
        return "\n".join(result)

In [ ]:
class TimezoneTool(Tool):
    name = "timezone"
    description = (
        "Get timezone information from a timezone name."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, timezone):
        url = "https://api.api-ninjas.com/v1/timezone"
        headers = {
            "X-Api-Key": self.api_key
        }
        params = {
            "timezone": timezone
        }
        response = requests.get(
            url,
            headers=headers,
            params=params
        )
        if response.status_code != 200:
            return f"Error: {response.text}"
        data = response.json()
        return (
            f"Timezone: {data['timezone']}\n"
            f"Local Time: {data['local_time']}\n"
            f"UTC Offset: {data['utc_offset']} seconds"
        )

In [ ]:
class EmailTool(Tool):
    name = "email"
    description = "Send an email using Resend."
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, to, subject, body):
        url = "https://api.resend.com/emails"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "from": "onboarding@resend.dev",
            "to": [to],
            "subject": subject,
            "html": f"<p>{body}</p>"
        }
        response = requests.post(
            url,
            headers=headers,
            json=payload
        )
        if response.status_code not in [200, 201]:
            return f"Error: {response.text}"
        data = response.json()
        return (
            "Email sent successfully!\n"
            f"Email ID: {data.get('id')}"
        )

In [ ]:
# Load API key
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
EXCHANGE_API_KEY = os.getenv("EXCHANGERATE_API_KEY")
NEWS_API_KEY = os.getenv("NEWSAPI_API_KEY")
NINJAS_API_KEY = os.getenv("APININJAS_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(WEATHER_API_KEY),
    "exchange_rate": ExchangeRateTool(EXCHANGE_API_KEY),
    "news": NewsTool(NEWS_API_KEY),
    "timezone": TimezoneTool(NINJAS_API_KEY),
    "email": EmailTool(RESEND_API_KEY)
}

In [ ]:
def execute_tool(tool_name, **kwargs):
    tool = TOOLS.get(tool_name)
    if not tool:
        return "Tool not found"
    return tool.run(**kwargs)

In [ ]:
result = execute_tool(
    "weather",
    city="Bangalore"
)
print(result)

In [ ]:
result = execute_tool(
    "exchange_rate",
    base_currency="USD",
    target_currency="INR"
)
print(result)

In [ ]:
result = execute_tool(
    "news",
    topic="Stocks",
    limit=3
)
print(result)

In [ ]:
result = execute_tool(
    "timezone",
    timezone="Europe/London"
)
print(result)

In [ ]:
result = execute_tool(
    "email",
    to="ogiboyinaakash@gmail.com",
    subject="Hello",
    body="This email was sent by my AI agent."
)
print(result)

### Multi-Step Task Execution

<small>Example task:  
"Book a flight and add it to my calendar."  

The agent may need:  
- Search flights  
- Compare prices  
- Choose best option  
- Book ticket  
- Generate confirmation  
- Add calendar event  
- Notify user  

This requires:  
- State tracking  
- Iterative planning  
- Tool orchestration  
- Failure handling </nsmall>

<small>**Planning Loop**  
while not task_completed:  
&emsp;think()  
&emsp;choose_action()  
&emsp;execute_tool()  
&emsp;observe_result()  
&emsp;update_plan()</nsmall>

#### Domain-specific Multi-Step Scenarios

<small>

**Finance: "Rebalance my portfolio to reduce tech exposure"**

The agent cannot know step 4 until it has executed step 3:

| Step | Action | Tool |
|---|---|---|
| 1 | Fetch current holdings | `get_portfolio` |
| 2 | Classify each holding by sector | `classify_sectors` |
| 3 | Identify overweight sectors (tech > 40%) | Reasoning only |
| 4 | Rank liquid positions for partial sale | `rank_by_liquidity` |
| 5 | **Request human approval** — "Sell 15% of NVDA?" | Approval gate |
| 6 | Execute trade if approved | `execute_order` |
| 7 | Log all decisions with timestamps | Decision log |

---

**Education: "Enrol me in the best Data Science track for my background"**

| Step | Action | Tool |
|---|---|---|
| 1 | Retrieve completed courses | `get_transcript` |
| 2 | Search available DS courses | `search_courses` |
| 3 | Check prerequisites for each candidate | `check_prerequisites` |
| 4 | Detect schedule conflicts | `check_schedule` |
| 5 | Rank by fit score | Reasoning only |
| 6 | **Request human confirmation** — "Enrol in ML-301?" | Approval gate |
| 7 | Submit enrolment + add to calendar | `enrol` + `create_event` |

---

**Healthcare: "Book the earliest cardiology slot that fits my schedule"**

| Step | Action | Tool |
|---|---|---|
| 1 | Retrieve patient availability | `get_patient_calendar` |
| 2 | Search open cardiology slots | `search_appointments` |
| 3 | Find earliest overlap | Reasoning only |
| 4 | **Require confirmation** — show proposed slot to patient | Approval gate |
| 5 | Confirm booking + send reminder | `book_appointment` + `send_reminder` |

---

**Key insight:** In all three cases the agent cannot write the flowchart before step 1 completes.  
That is the defining characteristic of a task that needs an agent rather than a chain.

</small>

#### Loop Control Mechanisms  

<small>Without loop control, agents can:   
- Run forever  
- Repeat useless actions  
- Spam APIs   
- Waste tokens  
- Enter failure cycles </nsmall>

**1. Max Iterations**  
<small>Limits total reasoning cycles.  

Example:  
MAX_ITERATIONS = k  
while step_count < MAX_STEPS:
    run_agent_step()
if step_count > MAX_ITERATIONS:  
&emsp;terminate("Task stopped: iteration limit exceeded")  

Purpose:  
- Prevent infinite loops  
- Control cost  
- Ensure predictable runtime</nsmall>

**2. Stopping Conditions**  
<small>Agent should stop when:  
- Goal achieved  
- Tool says “done”  
- Confidence is high  
- No more actions available  
- Repeated failures detected  

Example:   
if state["booking_confirmed"]:  
&emsp;break</nsmall>

### Error Recovery

<small>most beginner implementations:  
try:  
&emsp;tool.run()  
except:  
&emsp;pass  
This is NOT sufficient. 
 
Real agents must:  
- Diagnose failures   
- Retry intelligently  
- Switch strategies  
- Use fallback tools  
- Ask clarifying questions  
- Modify plans dynamically</nsmall>

<small>Types of tool failures
| Failure Type            | Example            |
| ----------------------- | ------------------ |
| Timeout                 | API takes too long |
| Invalid output          | Missing fields     |
| Rate limit              | Too many requests  |
| Tool unavailable        | Service down       |
| Hallucinated tool usage | Wrong parameters   |
| Parsing failure         | JSON malformed     |
| Empty result            | No search results  |
| Authentication error    | Invalid API key    |</nsmall>

<small>**Deliberate Failure**  
User asks: "Schedule a meeting tomorrow at 4 PM."    
Agent uses:  
- Calendar tool  
- Notification tool  

Calendar tool fails with:  
{  
&emsp;"error": "503 Service Unavailable"  
}  
A weak agent crashes.  
A strong agent recovers.</nsmall>

#### Recovery Logic

<small>**Detect Failure**  
result = calendar_tool.run(data)  
if "error" in result:  
&emsp;tool_failed = True</nsmall>

<small>**Failure classification**  
error_type = classify_error(result) 

Classifications:   
- TEMPORARY  
- INVALID_INPUT  
- AUTH_FAILURE  
- EMPTY_RESULT
- UNKNOWN</nsmall>

<small>**Recovery Strategies**  
| Error Type       | Recovery            |
| ---------------- | ------------------- |
| Timeout          | Retry               |
| Rate limit       | Backoff and retry   |
| Invalid input    | Reformat parameters |
| Empty result     | Alternative query   |
| Tool unavailable | Use fallback tool   |
| Auth error       | Escalate to user    |
| Unknown          | Abort safely        |</nsmall>

<small>**Retry Logic**  
Do not just implement: try again  

Instead do: 
- Limit retries  
- Change behavior  
- Add delay  
- Modify parameters  

Example: **exponential backoff**  
retry_count += 1  
if retry_count <= MAX_RETRIES:  
&emsp;wait(2 ** retry_count)</nsmall>

<small>**Fallback Tool Strategy**  
If primary calendar API fails:  
GoogleCalendarTool → OutlookCalendarTool  

Example: **real agent resilience**  
if google_failed:  
&emsp;use_outlook_calendar()

<small>**Re-Planning After Failure**   
The best agents re-plan dynamically. 

Example:  
Thought: Calendar API unavailable.  
Alternative approach: Store reminder locally and notify user.</nsmall>

<small>**Full Recovery Loop Example**

MAX_ITERATIONS = 5  
MAX_RETRIES = 2  

for iteration in range(MAX_ITERATIONS):  
&emsp;thought = agent.reason(state)  
&emsp;action = agent.choose_action(thought)  
&emsp;tool = tools[action["tool"]]  
&emsp;result = tool.run(action["input"])  
&emsp;# Failure handling  
&emsp;if "error" in result:  
&emsp;&emsp;error_type = classify_error(result)  
&emsp;&emsp;if error_type == "TEMPORARY":  
&emsp;&emsp;&emsp;retries += 1  
&emsp;&emsp;&emsp;if retries <= MAX_RETRIES:  
&emsp;&emsp;&emsp;&emsp;sleep(2 ** retries)    
&emsp;&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "INVALID_INPUT":  
&emsp;&emsp;&emsp;action["input"] = repair_input(action["input"])   
&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "TOOL_UNAVAILABLE":   
&emsp;&emsp;&emsp;tool = fallback_tools[action["tool"]]  
&emsp;&emsp;&emsp;result = tool.run(action["input"])  
&emsp;&emsp;else:  
&emsp;&emsp;&emsp;return "Task failed safely"  
&emsp;state.update(result)  
&emsp;if state["goal_completed"]:  
&emsp;&emsp;break   
</nsmall>

<small>**Agentic Recovery**  
detect_failure()  
classify_failure()  
choose_recovery_strategy()  
retry_or_replan()  
continue_execution()  

**Agent Structure**  
state = {  
&emsp;"goal": "...",  
&emsp;"completed_steps": [],  
&emsp;"failed_steps": [],  
&emsp;"tool_history": [],   
&emsp;"retry_counts": {},  
&emsp;"observations": [],  
&emsp;"current_plan": []  
}

**Structured Tool Output**  
{  
&emsp;"status": "success",  
&emsp;"data": {...}  
}  

**Tool History Tracking**  
tool_history.append({  
&emsp;"tool": "calendar",  
&emsp;"input": data,  
&emsp;"result": result  
})</nsmall>

### Agent Safety 

**Permission boundaries :**  
<small>They define what an agent is allowed to access or do.  

Without boundaries, an agent could:  
- delete files,  
- send emails,  
- execute dangerous commands,  
- leak confidential data,  
- make unauthorized purchases.</nsmall>

**Principle of Least Privilege :**  
<small>An agent should only receive:    
- the minimum tools,  
- minimum data,  
- minimum permissions needed for the task.  

Example:  
A calendar scheduling agent:  
- Can read calendar events  
- Can create meetings  
- Cannot access banking apps  
- Cannot execute shell commands</nsmall>

<small>Tool Access Categories
| Tool Type                | Access Level   | Example                             |
| ------------------------ | -------------- | ----------------------------------- |
| Read-only tools          | Safer          | Search API, documentation retrieval |
| Write tools              | Medium risk    | Email sender, database update       |
| Execution tools          | High risk      | Shell commands, Python execution    |
| Financial/critical tools | Very high risk | Payment systems, cloud deployment   |</nsmall>

**Tool Whitelisting :**  
<small>Agents should use only explicitly approved tools.  

ALLOWED_TOOLS = [  
&emsp;"search_docs",  
&emsp;"calendar_api",  
&emsp;"weather_api"  
]  
if requested_tool not in ALLOWED_TOOLS:  
&emsp;raise PermissionError("Tool access denied")</nsmall>

**Sandboxing :**  
<small>Running agents in isolated environments like:  
- containers,  
- virtual machines,  
- restricted APIs,  
- temporary credentials.  

This prevents system-wide damage if the agent behaves unexpectedly.</nsmall>

#### Loop Limits and Runaway Prevention

**Timeout Limits**  
<small>Prevent excessive execution time.  

import time  
start = time.time()  
if time.time() - start > Threshold:  
&emsp;stop_agent("Execution timeout")</nsmall>

**Duplicate Action Detection**  
<small>Prevent repeating same failed action.  

Example:  
if current_action in previous_actions:  
&emsp;repeated_count += 1  
if repeated_count > threshold:  
&emsp;stop_agent("Repeated action detected")</nsmall>

#### Approval steps for high-stakes Actions

<small>High-impact actions should require human approval before execution.  
This is known as Human-in-the-loop (HITL), confirmation gating or approval workflows.  

Examples:  
Sending external emails  
Deploying production code  
Financial transactions  
Deleting databases  
Modifying legal documents  
Accessing private records</nsmall>

**Approval Workflow** 

<small>**Agent Plans Action**  
Agent wants to: "Delete 500 inactive customer records"  

**Summary**  
Proposed Action:   
- Delete 500 records  
- Database: customers  
- Reason: inactive > 2 years  

**Human Approval**  
Approve? (yes/no)  

**Execute Only After Approval**  
if human_approved:  
&emsp;execute_action()  
else:   
&emsp;cancel()</nsmall>

**Logging** 
<small> 
| Event                   | Example                        |
| ----------------------- | ------------------------------ |
| User request            | "Book a meeting tomorrow"      |
| Agent reasoning summary | "Need calendar access"         |
| Tool calls              | `calendar.create_event()`      |
| Tool outputs            | "Meeting created successfully" |
| Errors                  | "API timeout"                  |
| Approval requests       | "Awaiting user confirmation"   |
| Final actions           | "Email sent"                   |

Structured Logging Example:  
{
&ensp;"timestamp": "2026-05-18T10:30:00",  
&emsp;"agent": "scheduler_agent",  
&emsp;"step": 4,  
&emsp;"action": "create_calendar_event",  
&emsp;"status": "success"  
}

Do NOT log:  
- hidden chain-of-thought,  
- sensitive credentials,  
- personal secrets,  
- raw authentication tokens  

Instead log:  
- concise reasoning summaries  
- action intents  
- decision outcomes</nsmall>

**Tool Failure Scenario**  
<small>An agent tries to email a report.  
email_api.send() → SMTP timeout  

**Unsafe Behavior**  
- Retry forever  
- Retry every second  
- Spam the API  

**Escalation**  
After repeated failures:  
- notify administrator
- log the issue
- pause execution
- request human intervention.

### Lab: Multi-Tool Agent Build

<small>Build a bounded research assistant that searches, summarizes, and saves a final note.  
Learners should implement the three tool bodies, the ReAct loop, error recovery, and a decision log.  
Use a local task boundary only: no unrestricted web browsing, no shell access, and no unapproved writes.</small>

<small>**Task scenario**  
A research assistant receives a question, searches a small approved knowledge source, summarizes the result, and saves the answer.  
The task is intentionally bounded so the agent can focus on planning, tool use, and recovery instead of open-ended browsing.  

**Rubric requirement**  
A valid solution must include:  
- explicit loop limits,  
- permission boundaries,  
- a decision log,  
- and a high-stakes approval gate for any write action.</small>

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path

class Tool(ABC):
    name = ""
    description = ""
    schema = {}
    REQUIRES_APPROVAL = False  # subclasses set True for high-stakes write actions

    @abstractmethod
    def run(self, **_kwargs):
        _ = _kwargs
        raise NotImplementedError


# Global intermittent failure state for demonstrations
RATE_LIMIT_STATE = {"counter": 0, "limit_every": 3}

class SearchTool(Tool):
    name = "search"
    description = "Search an approved local knowledge source."
    schema = {
        "name": "search",
        "description": "Search a bounded knowledge source and return ranked matches.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "minimum": 1, "maximum": 5},
            },
            "required": ["query"],
        },
    }

    def __init__(self, corpus_path):
        self.corpus_path = Path(corpus_path)

    def run(self, **kwargs):
        query = kwargs.get("query", "")
        top_k = kwargs.get("top_k", 3)
        q_lower = query.lower()

        # Deterministic simulated failures triggered by keywords
        if "timeout" in q_lower:
            return {"status": "error", "error_type": "timeout", "message": "simulated timeout"}
        if "rate_limit" in q_lower or "rate-limit" in q_lower:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated rate limit (keyword)"}
        if "malformed" in q_lower:
            return "THIS IS MALFORMED OUTPUT"

        # Intermittent rate-limit: flip on every Nth call to simulate transient throttling
        RATE_LIMIT_STATE["counter"] += 1
        if RATE_LIMIT_STATE["counter"] % RATE_LIMIT_STATE["limit_every"] == 0:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated intermittent rate limit"}

        matches = []
        for p in self.corpus_path.glob("*.txt"):
            text = p.read_text(encoding="utf-8")
            score = text.lower().count(query.lower())
            if score > 0:
                matches.append({"file": p.name, "score": score, "text": text})
        matches.sort(key=lambda x: x["score"], reverse=True)
        return {"status": "success", "results": matches[:top_k]}


class SummarizeTool(Tool):
    name = "summarize"
    description = "Summarize retrieved content into a short answer."
    schema = {
        "name": "summarize",
        "description": "Summarize search results into a concise response.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "style": {"type": "string", "enum": ["short", "bullet"]},
            },
            "required": ["text"],
        },
    }

    def run(self, **kwargs):
        text = kwargs.get("text", "")
        style = kwargs.get("style", "short")
        if style == "bullet":
            sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 20]
            summary = "\n".join(f"• {s}." for s in sentences[:4])
        else:
            summary = (text.split(".")[0].strip() + ".") if "." in text else text[:200]
        return {"status": "success", "summary": summary}


class SaveTool(Tool):
    """Write the final research note to disk.

    Flagged REQUIRES_APPROVAL = True because saving is a write action
    that persists beyond the agent session. The loop must gate this tool
    behind a human confirmation step before calling run().
    """
    name = "save"
    description = (
        "Save a research summary to a local note file. "
        "HIGH-STAKES write action — requires human approval before execution."
    )
    REQUIRES_APPROVAL = True
    schema = {
        "name": "save",
        "description": (
            "Save a research summary to the output folder. "
            "Requires explicit human approval."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "Target filename (no path)"},
                "content": {"type": "string", "description": "Text content to save"},
            },
            "required": ["filename", "content"],
        },
    }

    def run(self, **kwargs):
        filename = kwargs.get("filename", "research_note.txt")
        content = kwargs.get("content", "")
        out_dir = Path("output")
        out_dir.mkdir(exist_ok=True)
        # Restrict to output/ only — strip any path components to prevent traversal
        safe_name = Path(filename).name
        out_path = out_dir / safe_name
        out_path.write_text(content, encoding="utf-8")
        return {"status": "success", "saved_to": str(out_path), "bytes": len(content)}


# instantiate with local corpus
CORPUS_DIR = Path("data/C2.3 corpus").resolve()

TOOL_LIBRARY = {
    "search":    SearchTool(CORPUS_DIR),
    "summarize": SummarizeTool(),
    "save":      SaveTool(),
}

print("Tool library ready:", list(TOOL_LIBRARY.keys()))
for name, tool in TOOL_LIBRARY.items():
    approval_flag = "⚠ REQUIRES_APPROVAL" if tool.REQUIRES_APPROVAL else "✓ auto"
    print(f"  {name:12s} — {approval_flag} — {tool.description[:60]}")

<small>**ReAct loop requirements**  
The agent should alternate between reasoning, choosing one tool, observing the result, and updating state.  
It must stop when the task is complete, when the iteration limit is reached, or when repeated failures make progress unsafe.  

**Error recovery requirement**  
Do not use a bare try/except as the only answer.  
The agent should classify failures, retry with a change in behavior, re-plan when the output shape is wrong, and stop safely when recovery is not possible.</small>

In [ ]:
import time

MAX_ITERATIONS = 8
MAX_RETRIES    = 2
# Save is in the allowed set but gated behind approval — not excluded, just controlled
ALLOWED_TOOLS  = {"search", "summarize", "save"}

decision_log = []
RATE_LIMIT_STATE["counter"] = 0  # reset for clean demo


def classify_tool_result(result):
    if not isinstance(result, dict):
        return "unexpected_output"
    if result.get("status") == "error":
        return result.get("error_type", "unknown_error")
    if result.get("status") in {"success", "skipped"}:
        return "ok"
    return "unexpected_output"


def recovery_plan(error_type, retry_count):
    if error_type in {"timeout", "rate_limit"} and retry_count < MAX_RETRIES:
        return {"decision": "retry", "retry_count": retry_count + 1, "wait": 2 ** retry_count}
    if error_type == "unexpected_output":
        return {"decision": "replan", "retry_count": retry_count}
    return {"decision": "stop", "retry_count": retry_count}


def request_approval(tool_name, tool_input, simulated_answer=True):
    """
    In production use input() for real interactive approval.
    Here we use simulated_answer=True to allow the demo to run automatically.
    Set simulated_answer=False to demonstrate a denial.
    """
    print(f"\n[APPROVAL REQUIRED] tool={tool_name}")
    print(f"  Proposed input: {tool_input}")
    print(f"  Simulated decision: {'APPROVED' if simulated_answer else 'DENIED'}")
    return simulated_answer


def run_agent_step(state):
    """Deterministic planner: search → summarize → save (with approval)."""
    step = state.get("step", 1)
    query = state.get("query", "agent")

    if step == 1:
        state["step"] = 2
        return (
            f"Step 1 — Search corpus for '{query}'",
            {"tool": "search", "input": {"query": query, "top_k": 3}},
        )

    if step == 2:
        state["step"] = 3
        last = state.get("observations", [])[-1] if state.get("observations") else {}
        if not isinstance(last, dict) or last.get("status") != "success":
            # malformed or failed search — retry search
            state["step"] = 1
            return ("Step 2 — Retry search after bad result", {"tool": "search", "input": {"query": query, "top_k": 3}})
        texts = "\n".join(r.get("text", "") for r in last.get("results", []))
        return (
            "Step 2 — Summarize retrieved documents",
            {"tool": "summarize", "input": {"text": texts, "style": "bullet"}},
        )

    if step == 3:
        state["step"] = 4
        summary = ""
        for obs in reversed(state.get("observations", [])):
            if isinstance(obs, dict) and obs.get("status") == "success" and "summary" in obs:
                summary = obs["summary"]
                break
        return (
            "Step 3 — Save summary (requires human approval)",
            {"tool": "save", "input": {"filename": f"{query.replace(' ', '_')}_note.txt", "content": summary}},
        )

    return ("NoOp — goal already complete", {"tool": "search", "input": {"query": "noop", "top_k": 1}})


# ── Run the agent ────────────────────────────────────────────────────────────
state = {"goal_complete": False, "observations": [], "query": "agent safety"}
retry_count = 0
decision_log.clear()

for iteration in range(MAX_ITERATIONS):
    thought, action = run_agent_step(state)
    tool_name  = action["tool"]
    tool_input = action["input"]

    # ① Permission boundary
    if tool_name not in ALLOWED_TOOLS:
        decision_log.append({"iteration": iteration, "tool": tool_name, "status": "permission_denied"})
        print(f"[BLOCKED] {tool_name} is not in the allowed tool set.")
        break

    # ② Approval gate for high-stakes tools
    if TOOL_LIBRARY[tool_name].REQUIRES_APPROVAL:
        approved = request_approval(tool_name, tool_input, simulated_answer=True)
        if not approved:
            decision_log.append({"iteration": iteration, "tool": tool_name, "status": "approval_denied"})
            state["goal_complete"] = True          # treat denial as graceful stop
            state["final_status"]  = "save_declined_by_user"
            break
        decision_log.append({"iteration": iteration, "tool": tool_name, "status": "approval_granted"})

    # ③ Execute tool
    result = TOOL_LIBRARY[tool_name].run(**tool_input)
    decision_log.append({
        "iteration": iteration,
        "thought":   thought,
        "tool":      tool_name,
        "input":     tool_input,
        "result":    result,
    })

    # ④ Error classification & recovery
    result_state = classify_tool_result(result)
    if result_state != "ok":
        recovery = recovery_plan(result_state, retry_count)
        decision_log.append({"iteration": iteration, "recovery": recovery})
        if recovery["decision"] == "retry":
            retry_count = recovery["retry_count"]
            time.sleep(0)          # replace with time.sleep(recovery["wait"]) in production
            continue
        if recovery["decision"] == "replan":
            state["observations"].append({"repair_needed": result_state})
            continue
        state["final_status"] = "stopped_safely"
        break

    retry_count = 0
    state["observations"].append(result)

    # ⑤ Goal check
    if tool_name == "save" and result.get("status") == "success":
        state["goal_complete"] = True
        state["final_status"]  = "completed"
        break
else:
    state.setdefault("final_status", "stopped_by_iteration_limit")

# ── Decision log summary ─────────────────────────────────────────────────────
print(f"\nFinal status : {state.get('final_status', 'unknown')}")
print(f"Goal complete: {state['goal_complete']}")
print(f"\nDecision log ({len(decision_log)} entries):")
for entry in decision_log:
    i     = entry.get("iteration", "–")
    tool  = entry.get("tool", "–")
    st    = entry.get("status") or entry.get("result", {}).get("status", "–") if isinstance(entry.get("result"), dict) else entry.get("status", "–")
    rec   = entry.get("recovery", {}).get("decision", "") if entry.get("recovery") else ""
    thought_snippet = entry.get("thought", "")[:60]
    print(f"  [{i}] {tool:12s}  status={st:20s}  recovery={rec:8s}  thought={thought_snippet}")

<small>Interpretation: The agent searched the local corpus and matched `doc_001.txt` and `doc_003.txt`. The summarizer extracted the first sentence as a concise answer. The agent then attempted to save the note, but the save step is gated by human approval (not yet granted), demonstrating the safety boundary.</small>

#### LLM-driven ReAct Loop

<small>The deterministic loop above hard-codes the step sequence.  
A real agent uses the LLM to decide which tool to call next, based on what it has observed so far.  
The cell below implements a full Claude-powered ReAct loop with:

- tool selection driven by the model  
- approval gate for `save`  
- permission boundary enforced before every tool call  
- structured decision log at each step  

Swap `query` for any domain question to reuse the same loop across finance, education, healthcare, and e-commerce scenarios.</small>

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Build the tool schema list from the tool library
LLM_TOOL_SCHEMAS = [
    tool.schema for tool in TOOL_LIBRARY.values()
    if tool.schema
]

LLM_ALLOWED_TOOLS = {"search", "summarize", "save"}

SYSTEM_PROMPT = """You are a bounded research assistant.
Your only knowledge sources are the tools available to you.
Follow this process strictly:
1. Use the search tool to find relevant information from the approved knowledge base.
2. Use the summarize tool to condense the findings into a clear answer.
3. Use the save tool to persist the final answer (this requires human approval).
Always reason about what you found before choosing the next tool.
Stop as soon as the answer is saved or you cannot make further progress."""


def run_llm_react_agent(query, max_iterations=8, auto_approve_save=True):
    """
    Full LLM-driven ReAct loop using Claude tool use.

    Parameters
    ----------
    query            : The research question to answer.
    max_iterations   : Hard cap on reasoning cycles.
    auto_approve_save: Set False to simulate a user denying the save action.
    """
    messages = [{"role": "user", "content": query}]
    llm_decision_log = []

    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=LLM_TOOL_SCHEMAS,
            messages=messages,
        )

        # Capture any reasoning text the model produced
        reasoning = " | ".join(
            b.text for b in response.content if hasattr(b, "text") and b.text
        )
        log_entry = {
            "iteration":   iteration,
            "stop_reason": response.stop_reason,
            "thought":     reasoning,
        }

        # ── Model finished without needing more tools ──
        if response.stop_reason == "end_turn":
            llm_decision_log.append(log_entry)
            final_text = reasoning
            print(f"\n[DONE] Agent finished after {iteration + 1} iteration(s).")
            print(f"Answer: {final_text}")
            return {"answer": final_text, "decision_log": llm_decision_log}

        # ── Process tool calls ──
        tool_results = []
        for block in response.content:
            if not hasattr(block, "type") or block.type != "tool_use":
                continue

            tool_name  = block.name
            tool_input = block.input
            log_entry["tool"]  = tool_name
            log_entry["input"] = tool_input

            # ① Permission boundary
            if tool_name not in LLM_ALLOWED_TOOLS:
                tool_result = {"status": "error", "message": f"Tool '{tool_name}' is not permitted."}
                log_entry["permission"] = "denied"
                print(f"[BLOCKED] {tool_name} is outside the allowed tool set.")

            # ② Approval gate for high-stakes tools
            elif TOOL_LIBRARY.get(tool_name, Tool()).REQUIRES_APPROVAL:
                print(f"\n[APPROVAL REQUIRED] tool={tool_name}")
                print(f"  Filename : {tool_input.get('filename', '?')}")
                print(f"  Content  : {str(tool_input.get('content', ''))[:120]}...")
                granted = auto_approve_save
                print(f"  Decision : {'APPROVED' if granted else 'DENIED'}")
                log_entry["approval"] = "granted" if granted else "denied"
                if granted:
                    tool_result = TOOL_LIBRARY[tool_name].run(**tool_input)
                else:
                    tool_result = {"status": "skipped", "message": "Save declined by user."}

            # ③ Normal execution
            else:
                tool_result = TOOL_LIBRARY[tool_name].run(**tool_input)

            log_entry["result"] = tool_result
            print(f"  [{iteration}] {tool_name} → {str(tool_result)[:100]}")

            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(tool_result),
            })

        llm_decision_log.append(log_entry)

        # Feed tool results back to the model
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user",      "content": tool_results})

    return {
        "answer":       "Max iterations reached without completion.",
        "decision_log": llm_decision_log,
    }


# ── Domain scenario selector ─────────────────────────────────────────────────
# Change `query` to explore different domains against the corpus.

scenarios = {
    "general":     "What is an AI agent and how does the ReAct pattern work?",
    "finance":     "What safety boundaries should an AI agent have when handling financial transactions?",
    "education":   "Summarise how an AI agent can help a student plan their course sequence.",
    "healthcare":  "What approval and logging requirements apply to agents that schedule medical appointments?",
    "ecommerce":   "How should an agent handle tool failures when tracking a customer order?",
}

# Pick one scenario to run
query = scenarios["finance"]
print(f"Running agent for domain scenario: finance")
print(f"Query: {query}\n")

result = run_llm_react_agent(query, max_iterations=8, auto_approve_save=True)

print(f"\n{'='*60}")
print("Decision log:")
for entry in result["decision_log"]:
    i   = entry.get("iteration", "–")
    t   = entry.get("tool", "reasoning")
    st  = entry.get("result", {}).get("status", "–") if isinstance(entry.get("result"), dict) else "–"
    app = entry.get("approval", "")
    print(f"  [{i}] {t:12s}  status={st:10s}  approval={app}")

<small>**Deliberate failure scenario**  
The search tool may return a timeout, a rate limit response, or an unexpected shape such as a string instead of structured JSON.  
Learners should detect the failure type, retry only when it is temporary, re-plan when the output is malformed, and stop safely when the agent cannot make progress.</small>

<small>This output shows the tool wrapper returning a structured success payload.  
The `source` field tells us whether the result came from the live API or the local fallback, so the downstream agent can handle both cases with the same shape.</small>

In [3]:
# Demonstrate intermittent rate-limit by calling the search tool repeatedly
results = []
for i in range(1, 7):
    res = TOOL_LIBRARY['search'].run(query='agent', top_k=3)
    results.append((i, res))

for i, r in results:
    print(f"Call {i}: {r}")

Call 1: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more than one way, requiring clarification questions from the agent.'}, {'file': 'doc_001.txt', 'score': 1, 'text': 'AI agents are systems that combine reasoning and tool use. They follow the ReAct pattern: alternate thought and action while using tools such as search and email APIs.'}, {'file': 'doc_003.txt', 'score': 1, 'text': 'Error recovery should classify failures, retry intelligently, and re-plan when output is malformed. Agents must log decisions and enforce permission boundaries.'}]}
Call 2: {'status': 'error', 'error_type': 'rate_limit', 'message': 'simulated intermittent rate limit'}
Call 3: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more tha

<small>This trace shows the bounded agent pattern in action: it can search and summarize, but the write step is blocked until approval.  
That demonstrates the safety boundary and the decision log the rubric asks for.</small>

#### Lab Extension: Domain-specific Scenarios

<small>Use the same agent, tools, and corpus to explore how the same ReAct loop behaves across different professional domains.  
Swap the `query` variable in the LLM-driven loop cell above and observe how the tool selection changes.</small>

---

**Finance — Portfolio Risk Research**

```python
query = "What safety boundaries should an AI agent have when handling financial transactions?"
# Corpus file hit: finance_risk.txt, finance_portfolio.txt
# Expected path: search → summarize → save (with approval)
# Key teaching point: the save step is irreversible in a real system → approval gate is non-negotiable
```

---

**Education — Study Planning Assistant**

```python
query = "Summarise how an AI agent can help a student plan their course sequence."
# Corpus file hit: edu_course_planning.txt, edu_study_assistant.txt
# Expected path: search → summarize → save
# Key teaching point: the agent must present options before taking any enrolment action
```

---

**Healthcare — Scheduling Compliance**

```python
query = "What approval and logging requirements apply to agents that schedule medical appointments?"
# Corpus file hit: healthcare_scheduling.txt
# Key teaching point: every write action needs a timestamp, reason, and explicit confirmation
```

---

**E-commerce — Order Tracking with Recovery**

```python
query = "How should an agent handle tool failures when tracking a customer order?"
# Corpus file hit: ecommerce_agent.txt
# Key teaching point: loop limits prevent hammering a temporarily unavailable API
```

---

**Deliberate failure exercise**

```python
# Trigger a timeout and observe recovery
query = "timeout"          # keyword forces SearchTool to return a timeout error
# Expected: agent retries up to MAX_RETRIES, then stops safely

# Trigger malformed output
query = "malformed"        # keyword forces SearchTool to return a plain string instead of JSON
# Expected: agent replans after detecting unexpected_output
```